# 1-dars: Qoidadan o’rganishga — noldan raqam tanuvchi

**Bugun siz yozasiz:** qo’lyozma raqamlarni taniydigan dastur.
- tayyor model ham, ML kutubxonasi ham ishlatilmaydi — hammasini o’zingiz yozasiz;
- 1497 ta haqiqiy qo’lyozmadan o’rganadi va ~98% aniqlikka chiqadi;
- dars oxirida brauzerda raqam chizasiz, u **sizning** qo’lyozmangizni taniydi.

Dasturda bitta ham `if raqam == 7` bo’lmaydi.

⚠️ Avval: **File → Save a copy in Drive**. Keyin quyidagi katakni darhol ishga tushiring — u 5-bo’lim uchun kutubxona o’rnatadi (~1 daqiqa).

In [ ]:
!pip install -q gradio==6.30.0

## 1. Kompyuter uchun rasm — bu sonlar

Har bir rasm 8×8 = **64 ta son**: `0` — oq, `16` — to’liq qora.
`X` — rasmlar, `y` — har rasm aslida qaysi raqam.

In [ ]:
from sklearn.datasets import load_digits   # faqat ma'lumotni yuklash uchun
import random

raqamlar = load_digits()
X = raqamlar.data.tolist()     # 1797 ta rasm, har biri 64 ta son
y = raqamlar.target.tolist()   # har rasm qaysi raqam: 0..9

print(len(X), "ta rasm")
print("Birinchi rasm:", X[0])
print("Bu raqam:", y[0])

In [ ]:
import matplotlib.pyplot as plt

def jadval(rasm):
    """64 ta sonni 8 qatorda chiqaradi."""
    for q in range(8):
        print(" ".join(f"{int(p):2}" for p in rasm[q * 8:(q + 1) * 8]))

def korsat(rasm, sarlavha=""):
    plt.figure(figsize=(2, 2))
    plt.imshow([rasm[q * 8:(q + 1) * 8] for q in range(8)], cmap="gray_r")
    plt.title(sarlavha)
    plt.axis("off")
    plt.show()

jadval(X[0])
korsat(X[0], f"y = {y[0]}")

In [ ]:
# TODO: datasetdan birinchi "7" ni toping (y dan uning indeksini),
# uni jadval() va korsat() bilan chiqaring. Sonlarda 7 shaklini ko'ryapsizmi?


**Train va test.** Rasmlarni aralashtirib, 300 tasini chetga olib qo’yamiz. Bu — **test**: model ularni o’qishda ko’rmaydi. Faqat oxirida “imtihon” uchun ishlatiladi.

In [ ]:
juftlar = list(zip(X, y))
random.Random(42).shuffle(juftlar)
test, train = juftlar[:300], juftlar[300:]

X_train = [x for x, _ in train]
y_train = [t for _, t in train]
X_test = [x for x, _ in test]
y_test = [t for _, t in test]
print("Train:", len(X_train), "| Test:", len(X_test))

## 2. Qoida bilan: 0 va 1 ni ajrating

`0` ning o’rtasi bo’sh, `1` ning o’rtasi to’la. Markazdagi piksellarga qarab qoida yozing.
`rasm[qator * 8 + ustun]` — shu katakdagi son (qator va ustun 0 dan 7 gacha).

In [ ]:
def aniqlik(togri, bashorat):
    """To'g'ri topilganlar ulushi, foizda."""
    mos = sum(1 for t, b in zip(togri, bashorat) if t == b)
    return 100 * mos / len(togri)


nol_bir = [(x, t) for x, t in zip(X_test, y_test) if t in (0, 1)]
print(len(nol_bir), "ta 0 va 1 rasmi")

In [ ]:
def qoida_0_1(rasm):
    # TODO: markazdagi piksellar yig'indisini hisoblang.
    # Katta bo'lsa — 1, kichik bo'lsa — 0. Chegarani o'zingiz tanlang.
    ...


qoida_bashorat = [qoida_0_1(x) for x, _ in nol_bir]
print(f"0/1 qoidasi: {aniqlik([t for _, t in nol_bir], qoida_bashorat):.0f}%")

Ikki raqam uchun ishladi. Endi **10 ta raqam** uchun qoida yozib ko’ring: 3 va 8 ni qanday ajratasiz? 4 va 9 ni-chi? Har kim har xil yozadi: qiyshiq, qalin, kichik...

Qoida bilan bu yo’l berk. Boshqacha qilamiz: **qoidani yozmaymiz, misollardan o’rganamiz.**

## 3. Eng yaqin qo’shni (1-NN) — noldan

G’oya: yangi rasm train’dagi **qaysi rasmga eng o’xshash** bo’lsa — o’sha raqam deb aytamiz.
“O’xshash” = masofa kichik. 2 ta son uchun Pifagor: $\sqrt{(a_1-b_1)^2 + (a_2-b_2)^2}$. 64 ta son uchun ham xuddi shunday, faqat qo’shiluvchi 64 ta:
$\sqrt{(a_1-b_1)^2 + \dots + (a_{64}-b_{64})^2}$

In [ ]:
import math

def masofa(a, b):
    # TODO: a va b — bir xil uzunlikdagi sonlar ro'yxati.
    # Har juft son farqining kvadratini qo'shing, oxirida ildiz oling.
    # Yordam: for p, q in zip(a, b): ...
    ...


assert masofa([0, 0], [3, 4]) == 5
assert masofa([1, 2, 3], [1, 2, 3]) == 0
assert masofa([0] * 64, [1] * 64) == 8
print("masofa() ishlayapti ✅")

In [ ]:
def bashorat_1nn(rasm, X, y):
    # TODO: X dagi har bir rasmgacha masofani hisoblang.
    # Eng yaqinining javobini (y dan) qaytaring.
    eng_kichik = float("inf")   # hozircha eng kichik masofa — cheksiz
    javob = None
    ...
    return javob


assert bashorat_1nn(X_train[0], X_train, y_train) == y_train[0]   # o'zi o'ziga eng yaqin
print("bashorat_1nn() ishlayapti ✅")

**Imtihon:** model ko’rmagan 300 ta rasm. Natijangizni mentorga ayting — doskaga yoziladi.

In [ ]:
import time

bosh = time.time()
bashoratlar = [bashorat_1nn(x, X_train, y_train) for x in X_test]
print(f"Aniqlik: {aniqlik(y_test, bashoratlar):.1f}%   | vaqt: {time.time() - bosh:.1f} soniya")

Siz hozir qo’lyozma raqamlarni taniydigan dastur yozdingiz. Unda bitta ham raqam haqidagi qoida yo’q — hamma “bilim” **ma’lumotda**.

## 4. Model qayerda adashdi?

Professional ML muhandisi birinchi bo’lib aniqlikka emas, **xatolarga** qaraydi.

In [ ]:
xatolar = [(x, t, b) for x, t, b in zip(X_test, y_test, bashoratlar) if t != b]
print(len(xatolar), "ta xato.  Sarlavha: haqiqatda → model aytgani")

plt.figure(figsize=(14, 2))
for i, (x, t, b) in enumerate(xatolar[:10]):
    plt.subplot(1, 10, i + 1)
    plt.imshow([x[q * 8:(q + 1) * 8] for q in range(8)], cmap="gray_r")
    plt.title(f"{t} → {b}")
    plt.axis("off")
plt.show()

Siz bularni to’g’ri tanirmidingiz? Qaysi xato “kechirarli”, qaysi biri g’alati?

## 5. O’z qo’lyozmangiz

Quyidagi ilovada raqam chizing. O’ngda model ko’radigan 8×8 rasm chiqadi.
- Modelni “qiynang”: qiyshiq, juda qalin, juda ingichka, o’zgacha shaklda yozing. Sichqoncha bilan yozish qog’ozdagidek emas — model albatta qaysidir raqamda adashadi.
- Adashgan raqamni toping. To’g’ri raqamni tanlang va **“Train’ga qo’shish”** ni bosing. Shu raqamni o’z uslubingizda 3–5 marta chizib qo’shing va qayta sinang.
- Juftingiz bilan almashing: sizning yozuvingizni o’rgangan model uning yozuvini taniydimi?

`rasmni_8x8()` funksiyasini hozir tushunish shart emas. Uni 2-darsda numpy bilan o’zingiz yozasiz.

In [ ]:
# Tayyor yordamchi: chizmani datasetdagidek 8×8 ga o'tkazadi
import numpy as np
from PIL import Image

def rasmni_8x8(chizma):
    if isinstance(chizma, dict):                 # Sketchpad qiymati
        chizma = chizma.get("composite")
    if chizma is None:
        return None
    a = np.asarray(chizma).astype(float)
    if a.ndim == 3:
        siyoh = 255 - a[..., :3].mean(axis=2)
        if a.shape[2] == 4:                      # shaffof fon
            siyoh = siyoh * a[..., 3] / 255
    else:
        siyoh = 255 - a
    q, u = np.where(siyoh > 50)
    if len(q) == 0:
        return None
    siyoh = siyoh[q.min():q.max() + 1, u.min():u.max() + 1]   # raqam atrofini kesish
    h, w = siyoh.shape
    t = max(h, w)
    kvadrat = np.zeros((t, t))
    kvadrat[(t - h) // 2:(t - h) // 2 + h, (t - w) // 2:(t - w) // 2 + w] = siyoh
    b = np.asarray(Image.fromarray(kvadrat.astype(np.uint8)).resize((32, 32), Image.BILINEAR)) > 40
    for _ in range(3):                           # chiziq juda ingichka bo'lsa — qalinlashtirish
        if b.reshape(8, 4, 8, 4).sum(axis=(1, 3)).mean() >= 4:
            break
        c = b.copy()
        c[1:] |= b[:-1]; c[:-1] |= b[1:]; c[:, 1:] |= b[:, :-1]; c[:, :-1] |= b[:, 1:]
        b = c
    return b.reshape(8, 4, 8, 4).sum(axis=(1, 3)).flatten().astype(float).tolist()


def korinish(rasm):
    """8×8 rasmni ekranda ko'rinadigan kattalikka o'tkazadi."""
    m = 255 - np.array(rasm).reshape(8, 8) * 255 / 16
    return np.kron(m, np.ones((20, 20))).astype(np.uint8)

In [ ]:
import gradio as gr

def tani(chizma):
    rasm = rasmni_8x8(chizma)
    if rasm is None:
        return "Avval raqam chizing", None
    return f"Bu — {bashorat_1nn(rasm, X_train, y_train)}", korinish(rasm)

def qosh(chizma, togri):
    rasm = rasmni_8x8(chizma)
    if rasm is None:
        return "Avval raqam chizing"
    X_train.append(rasm)
    y_train.append(int(togri))
    return f"Qo'shildi: {togri}. Train'da endi {len(X_train)} ta rasm"

with gr.Blocks() as ilova:
    gr.Markdown("### Raqam chizing")
    with gr.Row():
        chizma = gr.Sketchpad(type="numpy", canvas_size=(280, 280))
        with gr.Column():
            javob = gr.Textbox(label="Model javobi")
            kordi = gr.Image(label="Model ko'rgan 8×8", height=160)
    gr.Button("Tanib ber", variant="primary").click(tani, chizma, [javob, kordi])
    with gr.Row():
        togri = gr.Dropdown([str(i) for i in range(10)], value="7", label="Aslida qaysi raqam?")
        holat = gr.Textbox(label="Holat")
    gr.Button("Train'ga qo'shish").click(qosh, [chizma, togri], holat)

ilova.launch()

Model sizning yozuvingizda adashdi. Siz uni qanday tuzatdingiz? **Kodni o’zgartirmadingiz — ma’lumot qo’shdingiz.**

> **Oddiy dastur:** odam qoida yozadi → xato bo’lsa, kod o’zgaradi.
> **ML:** model misollardan o’rganadi → xato bo’lsa, **ma’lumot** o’zgaradi.

## 6. Yakun

**Saqlash:** File → Save a copy in GitHub → repozitoriy `pdp-ai-kurs`, yo’l `1-oy/01-dars/amaliyot.ipynb`.

**2–3 gapda yozing:** sizning dasturingiz qanday qilib raqamni taniydi? Nega 10 ta raqam uchun qoida yozish o’rniga shunday qildik?

*Javobingiz:* ...

---
# Uy ishi

### 1. AI yozgan kodni tekshirish
1 ta emas, **k ta** eng yaqin qo’shni ovoz bersa, model ishonchliroq bo’lishi mumkin. Ovoz sanash funksiyasini AI yozib berdi — u bitta testdan o’tadi.
**Professional qoida:** AI yozgan kodni testlar bilan tekshirmasdan ishlatmaymiz. Yana kamida 3 ta test yozing. Xato topsangiz, tuzating.

In [ ]:
def kop_uchraydigan(belgilar):
    """Ro'yxatda eng ko'p uchraydigan belgini qaytaradi. (AI yozgan)"""
    hisob = {}
    for b in belgilar:
        hisob[b] = hisob.get(b, 0) + 1
    return max(hisob)


assert kop_uchraydigan([8, 8, 3]) == 8
# TODO: yana kamida 3 ta test
print("Testlar o'tdi ✅")

### 2. k-NN: k = 1, 3, 5, 7
`bashorat_knn()` ni yozing va har bir `k` uchun aniqlikni jadvalga yozing. Qaysi `k` eng yaxshi?
⚠️ `k` ni test natijasiga qarab tanlash — yashirin xato. Nega ekanini 4-darsda ko’ramiz.

Agar 5-bo’limda train’ga rasm qo’shgan bo’lsangiz, avval 1-bo’limdan boshlab qayta ishga tushiring: test datasetning o’zidan bo’lsin.

In [ ]:
def bashorat_knn(rasm, X, y, k=3):
    # TODO: (masofa, belgi) juftliklarini masofa bo'yicha tartiblang,
    # birinchi k tasining belgilarini oling va kop_uchraydigan() bilan ovoz bering
    ...


# for k in [1, 3, 5, 7]:
#     b = [bashorat_knn(x, X_train, y_train, k) for x in X_test]
#     print(k, aniqlik(y_test, b))

### 3. Qaysi raqamlar ko’p adashtiriladi?
`xatolar` ro’yxatidan `(haqiqiy, bashorat)` juftliklarini sanang (lug’at bilan). Eng ko’p uchraydigan 3 juftlikni chiqaring va 3 gap yozing: nega aynan shular?

In [ ]:
# TODO


### 4. Professional kutubxona bilan solishtirish
Xuddi shu modelni `sklearn` bilan quring: `KNeighborsClassifier(n_neighbors=1)`, `.fit(...)`, `.predict(...)`.
Bashoratlar sizniki bilan bir xilmi? Qaysi biri necha marta tezroq?

In [ ]:
# TODO


### Chuqurroq
- `masofa()` o’rniga “Manhattan masofa”ni sinang: `sum(abs(p - q) ...)`. Aniqlik o’zgaradimi?
- Bizning dastur 300 ta rasmni bir necha soniyada tanidi. 1 million rasm bo’lsa-chi? 2-darsda numpy bilan shu hisobni 100 marta tezlashtiramiz.